In [ ]:
%%capture
!pip install -U openai pydantic langchain langchain-community langchain-openai datasets fsspec faiss-gpu sentence_transformers

In [ ]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter Your OpenAI API Key:")

Enter Your OpenAI API Key:··········


## Self-Consistency Prompting in Language Models

Self-consistency prompting was introduced in the March 2022 paper ["Self-Consistency Improves Chain of Thought Reasoning in Language Models"](https://arxiv.org/pdf/2203.11171.pdf) by Xuezhi Wang, et. al.

# 🤔 **What is Self-Consistency Prompting?**

- Focuses on exploring different reasoning paths for complex problems.

- Aims for reliable answers by checking consistency across various thought processes.

### 🌟 **Differences from Other Prompting Techniques**

- Traditional CoT: Generates short sentences mimicking human reasoning for solving tasks.

- Self-Consistency: Samples multiple reasoning paths and finds the most consistent answer.

- It's unsupervised and works with pre-trained models, unlike methods needing extra training or human annotations.

### **🛠️ Constructing a Self-Consistency Prompt**


<figure>
  <img src="https://idevtek.com/images/Self-Consistency.jpg" alt="Image Description" style="width:100%">
  <figcaption>The self-consistency method contains three steps: (1) prompt a language model using chain-of-thought (CoT) prompting; (2) replace the “greedy decode” in CoT prompting by sampling from the language model’s decoder to generate a diverse set of reasoning paths; and (3) marginalize out the reasoning paths and aggregate by choosing the most consistent answer in the final answer set.</figcaption>
  <a href="https://twitter.com/denny_zhou/status/1584978661668966401">Image Source</a>
</figure>


1. **Start with CoT**: Use chain-of-thought prompting as the base.

2. **Sample Diverse Paths**: Use "sample-and-marginalize" decoding to get various reasoning paths.

3. **Marginalize for Consistency**: Find the most consistent answer from these different paths.

### 🔍 **Examples in Action**

- **Chain-of-Thought**: For the question "If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?", instead of directly answering "5", the model might respond with the reasoning: "There are 3 cars in the parking lot already. 2 more arrive. Now there are 3 + 2 = 5 cars. The answer is 5."

- **Self-Consistency**: For a question about how much Janet makes from selling eggs, the model might generate multiple reasoning paths like:
  1. "She has 16 - 3 - 4 = 9 eggs left. So she makes $2 * 9 = $18 per day."

  2. "This means she sells the remainder for $2 * (16 - 4 - 3) = $26 per day."
  
  3. "She eats 3 for breakfast, so she has 16 - 3 = 13 left. Then she bakes muffins, so she has 13 - 4 = 9 eggs left. So she has 9 eggs * $2 = $18."

  The model then aggregates these paths to determine the most consistent answer, which in this case is "$18 per day."


🚀 **Impact of Self-Consistency Prompting**
- Enhances model reasoning by considering multiple paths.
- Shown to boost performance in arithmetic and commonsense reasoning tasks.

# Begin by setting up CoT prompts

We're following the same pattern from the Chain of Thought lesson.

1. Downloading CoT prompt datasets from HuggingFace

2. Downloading embeddings model from HuggingFace

3. Creating prompt template for CoT

4. Creating an example selector

5. Construct the prompt

In [ ]:
import json
import pandas as pd
import requests

# 1. Download the JSON data from Hugging Face
url = "https://skillbakery.com/sb-dev/cot-coll.json"
response = requests.get(url)

# 2. Check for download success
if response.status_code != 200:
    raise Exception(f"Failed to download dataset! Status code: {response.status_code}")

# 3. Parse JSON content
data = response.json()

# 4. Convert to a list of dictionaries with 'source', 'rationale', and 'target' keys
selected_examples = []
for key, value in data.items():
    if 'source' in value and 'rationale' in value and 'target' in value:
        selected_examples.append({
            'source': value['source'],
            'rationale': value['rationale'],
            'target': value['target']
        })

# 5. Shuffle and select top 10,000 examples
import random
random.seed(42)
random.shuffle(selected_examples)
selected_examples = selected_examples[:10000]


# 6. Preview
print(selected_examples[0])

{'source': 'Two analogies that relate actions to the tools used to perform the action is given in the form "A : B. C : ?". "A : B" relates action A to tool B. Your task is to replace the question mark (?) with the appropriate tool for the given action C, following the "A : B" relation.\n\nvacuum : vacuum. drill : ?', 'rationale': 'To vacuum the floor, you would use a vacuum. Similarly, to drill a hole in the wall, you would use a drill.', 'target': 'drill'}


In [ ]:
!pip install -U openai pydantic langchain langchain-community langchain-openai datasets fsspec faiss-cpu sentence_transformers langchain-huggingface

from langchain_huggingface import HuggingFaceEmbeddings

model_name = "BAAI/bge-base-en-v1.5"

encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={'device': 'cpu'},
    encode_kwargs=encode_kwargs
)

  Using cached fsspec-2025.7.0-py3-none-any.whl.metadata (12 kB)


In [ ]:
from langchain.prompts.example_selector import MaxMarginalRelevanceExampleSelector
from langchain_community.vectorstores import FAISS
from langchain.prompts import FewShotPromptTemplate, PromptTemplate

prefix = "Consider the following as examples of how to reason:"

examples_template = """Query: {source}

Rationale: {rationale}

Response: {target}
"""

suffix = """Using a similar reasoning approach, answer the users question which is delimited by triple backticks.

User question: ```{input}```

Take a deep breath, break down the user's query step-by-step, and provide a clear chain of thought in your response."
"""

examples_prompt = PromptTemplate(
    input_variables=["source", "target"],
    template=examples_template
)

In [ ]:
example_selector = MaxMarginalRelevanceExampleSelector.from_examples(
    selected_examples,
    embeddings,
    FAISS,
    k=5,
)

mmr_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=examples_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["input"]
)

In [ ]:
query = """There were nine computers in the server room. Five more computers were installed each day, from
monday to thursday. How many computers are now in the server room?
"""
prompt = mmr_prompt.format(input=query)

In [ ]:
print(prompt)

Consider the following as examples of how to reason:

Query: In this task, you need to count the number of words in a sentence that end with the given letter. Answer with numbers and not words.

Sentence: 'a kitchen with a table dishwasher sink and microwave'. How many words end with the letter 'h' in the sentence.

Rationale: The sentence is 'a kitchen with a table dishwasher sink and microwave'. There are 9 words in total.\n1. 'a' -> (total) 1\n2. 'kitchen' -> (total) 2\n3. 'with' -> (total) 3\n4. 'a' -> (total) 4\n5. 'table' -> (total) 5\n6. dishwasher -> 6, No match since it ends with the letter "r", not "h". \t(Total remains unchanged)\n7. sink: 7, No match since it ends with the letter "k", not "h". \t(Total remains unchanged)\n8.'and': 8, No match since it ends with the letter "d", not "h". \t(Total remains unchanged)\n9.'microwave': 9, This word matches! Add 1 to total to get 10! The final answer is 10 - 9 = 1

Response: 1


Query: Write the conversation response. I need to buy

# Self-Consistency Prompt

In [ ]:
sc_template = """Based on the responses (delimited by < >) to the following query, \
(delimited by triple backticks) return the response that occurs most frequently.

Query: ```{query}```

Responses: <{responses}>
"""

sc_prompt = PromptTemplate.from_template(sc_template)

### Generating multiple responses


Use the `n` parameter to generate alternative responses. Increase `n` to explore different variations.



In [ ]:
from langchain_openai import OpenAI

# we'll use the default model here, gpt-3.5-turbo-instruct
llm = OpenAI(n=5)

generations = llm.generate([prompt])

In [ ]:
generations.generations

[[Generation(text="\nRationale: The user's question is asking for the total number of computers in the server room after five more computers were installed each day from Monday to Thursday. To answer this, we need to first determine how many days have passed since Monday. Since it is mentioned that five more computers were installed each day, we can infer that four days have passed. Therefore, we can calculate the total number of computers by adding 9 (initial number of computers) and 4*5 (5 more computers were installed each day for 4 days). This gives us a total of 29 computers in the server room.\n\nResponse: There are 29 computers in the server room now.", generation_info={'finish_reason': 'stop', 'logprobs': None}),
  Generation(text='\nRationale: The user is asking how many computers are now in the server room after five more were installed each day from Monday to Thursday. To answer this, we need to add the initial number of computers (9) with the total number of computers insta

In [ ]:
responses = []
for item in generations.generations[0]:
    response_index = item.text.find("Response: ")
    if response_index != -1:
        response = item.text[response_index + len("Response: "):].strip()
        responses.append(response)

In [ ]:
responses

['There are now 29 computers in the server room.',
 'There are now 20 computers in the server room.',
 'There are currently 29 computers in the server room.',
 'There are now 24 computers in the server room.',
 'There are currently 29 computers in the server room.']

In [ ]:
llm = OpenAI()

final_prompt = sc_prompt.format(query=query, responses=str(responses))

print(final_prompt)

Based on the responses (delimited by < >) to the following query, (delimited by triple backticks) return the response that occurs most frequently.

Query: ```There were nine computers in the server room. Five more computers were installed each day, from
monday to thursday. How many computers are now in the server room?
```

Responses: <['There are now 29 computers in the server room.', 'There are now 20 computers in the server room.', 'There are currently 29 computers in the server room.', 'There are now 24 computers in the server room.', 'There are currently 29 computers in the server room.']>



In [ ]:
print(llm.invoke(final_prompt))


The response that occurs most frequently is: There are currently 29 computers in the server room.
